In [1]:
from pathlib import Path

from sentence_transformers import SentenceTransformer
from config import config
import random, numpy as np, pandas as pd, os
from transformers import AutoTokenizer, AutoModel
import torch

from dotenv import load_dotenv

d:\MSc\3. Spring 2026\CSE715\Project\vae-audio-clustering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv() 
token = os.environ.get("HF_TOKEN", None)
random.seed(42)

In [3]:
root = Path("../..")

In [4]:
LYRICS_DIR_EN = root / config.LYRICS_DIR_EN
LYRICS_DIR_BN = root / config.LYRICS_DIR_BN
EMBEDDINGS_DIR = root / config.EMBEDDINGS_DIR
METADATA_EN = root / config.METADATA_DIR / "metadata_en.csv"
METADATA_BN = root / config.METADATA_DIR / "bn" / "metadata_bn.csv"

In [5]:
LYRICS_DIR_EN.exists(), LYRICS_DIR_BN.exists(), METADATA_EN.exists(), METADATA_BN.exists()

(True, True, True, True)

In [6]:
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
model = SentenceTransformer('all-mpnet-base-v2', token=token)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6136.26it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
bn_tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert", token=token)
bn_model = AutoModel.from_pretrained("csebuetnlp/banglabert", token=token)
bn_model.eval()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 41877.14it/s]
ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ElectraModel(
  (embeddings): ElectraEmbeddings(
    (word_embeddings): Embedding(32000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): ElectraEncoder(
    (layer): ModuleList(
      (0-11): 12 x ElectraLayer(
        (attention): ElectraAttention(
          (self): ElectraSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): ElectraSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0

In [9]:
def encode_banglabert(text: str) -> np.ndarray:
    inputs = bn_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    with torch.no_grad():
        outputs = bn_model(**inputs)
    # Mean-pool the last hidden state across tokens
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

In [10]:
lyric_stems = [
    p.stem for p in LYRICS_DIR_BN.iterdir()
    if p.is_file() and p.suffix.lower() == ".txt"
]

In [11]:
df_meta_bn = pd.read_csv(METADATA_BN)
relevant_meta_bn = df_meta_bn[df_meta_bn["id"].isin(lyric_stems)]

In [12]:
relevant_meta_bn.shape

(125, 3)

In [13]:
ok, skipped, errors = 0, 0, 0

for lyric_stem in lyric_stems:
    row = relevant_meta_bn[relevant_meta_bn["id"] == lyric_stem]

    if row.empty:
        print(f"  [SKIP] {lyric_stem} — not found in metadata")
        skipped += 1
        continue

    song_id = str(row["id"].item())
    out_path = EMBEDDINGS_DIR / f"{song_id}.npy"

    if out_path.exists():
        print(f"  [EXISTS] {song_id} — already embedded, skipping")
        skipped += 1
        continue

    try:
        lyric_path = LYRICS_DIR_BN / f"{lyric_stem}.txt"
        with open(lyric_path, "r", encoding="utf-8") as f:
            text = f.read()

        z_text = encode_banglabert(text)
        np.save(out_path, z_text)
        print(f"  [OK] {song_id}")
        ok += 1

    except Exception as e:
        print(f"  [ERROR] {song_id}: {e}")
        errors += 1

print(f"  → Done: {ok} saved, {skipped} skipped, {errors} errors")

  [OK] -Bfa-aVa7MU_seg_00002
  [OK] -Bfa-aVa7MU_seg_00004
  [OK] -Bfa-aVa7MU_seg_00005
  [OK] -ph4mykFp9I_seg_00005
  [OK] -ph4mykFp9I_seg_00007
  [OK] -ph4mykFp9I_seg_00008
  [OK] -ph4mykFp9I_seg_00015
  [OK] -ph4mykFp9I_seg_00016
  [OK] 8dLOs8LK4OQ_seg_00001
  [OK] 8dLOs8LK4OQ_seg_00002
  [OK] 8dLOs8LK4OQ_seg_00003
  [OK] 8dLOs8LK4OQ_seg_00004
  [OK] 8dLOs8LK4OQ_seg_00011
  [OK] 8dLOs8LK4OQ_seg_00012
  [OK] 8dLOs8LK4OQ_seg_00014
  [OK] 8dLOs8LK4OQ_seg_00016
  [OK] aOJYulVhCWk_seg_00001
  [OK] aOJYulVhCWk_seg_00002
  [OK] aOJYulVhCWk_seg_00009
  [OK] BWIW9aBNmlk_seg_00001
  [OK] BWIW9aBNmlk_seg_00002
  [OK] BWIW9aBNmlk_seg_00003
  [OK] BWIW9aBNmlk_seg_00004
  [OK] BWIW9aBNmlk_seg_00007
  [OK] BWIW9aBNmlk_seg_00011
  [OK] BWIW9aBNmlk_seg_00013
  [OK] BWIW9aBNmlk_seg_00014
  [OK] CKfhGvUPXkY_seg_00005
  [OK] CKfhGvUPXkY_seg_00007
  [OK] CKfhGvUPXkY_seg_00008
  [OK] CKfhGvUPXkY_seg_00009
  [OK] CKfhGvUPXkY_seg_00011
  [OK] cWlAszeTFG0_seg_00002
  [OK] cWlAszeTFG0_seg_00004
  [OK] cWlAsze

In [14]:
en_lyric_list = [p.stem for p in LYRICS_DIR_EN.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]

In [15]:
df_meta_en = pd.read_csv(METADATA_EN)

In [16]:
relevant_meta_en = df_meta_en[df_meta_en['lyric_file_stem'].isin(en_lyric_list)]

In [17]:
for lyric_path in en_lyric_list:
    song_id = relevant_meta_en[relevant_meta_en["lyric_file_stem"]==lyric_path]["audio_file_stem"].item()
    song_id = str(song_id)
    print(song_id)
    
    try:
        main_file_path = LYRICS_DIR_EN / f"{lyric_path}.txt"
        if main_file_path.exists():
            with open(main_file_path, 'r', encoding='utf-8') as f:
                text = f.read()
            
            # Generate vector (384 dimensions)
            z_text = model.encode(text)
            
            # Save as .npy using the song_id as name
            np.save(EMBEDDINGS_DIR / f"{song_id}.npy", z_text)
        else:
            print(f"Warning: File not found at {main_file_path}")
            
    except Exception as e:
        print(f"Error processing {song_id}: {e}")

A120-168
A151-172
A061-66
A167
A168
MT0000092637
MT0000133200
MT0000174580
MT0000185601
MT0000604234
MT0000615489
MT0000619767
MT0000734482
MT0000764951
MT0000789330
MT0001109401
MT0001383277
MT0001395462
MT0001511433
MT0001514284
MT0001566152
MT0001795697
MT0001841416
MT0001889075
MT0001889244
MT0001972829
MT0002222944
MT0002238737
MT0002239079
MT0003117846
MT0003131369
MT0003213617
MT0003274067
MT0003287931
MT0003501434
MT0003770678
MT0003774577
MT0003881997
MT0004362501
MT0004476235
MT0004506984
MT0004625127
MT0005030845
MT0005567133
MT0005916227
MT0006159067
MT0006196650
MT0006357085
MT0006357849
MT0006367176
MT0006406298
MT0006475078
MT0006737689
MT0007087238
MT0007113915
MT0007210245
MT0007280044
MT0007349422
MT0007828167
MT0008241153
MT0008296972
MT0008960612
MT0009075362
MT0009239443
MT0009385684
MT0009635397
MT0009968194
MT0010261584
MT0010261588
MT0010310891
MT0010310899
MT0010623104
MT0010674859
MT0010857270
MT0011358600
MT0011378875
MT0011387901
MT0011870505
MT0011908274
MT